# DodoGo - Google Maps to OpenStreetMap Migration Case (NB05)

This notebook supports Sections 2.5 and 3.5 of the thesis. It documents the move from a Google Maps dependent stack to an OpenStreetMap-based approach. The public repository version uses relative cost indexes instead of exact billing values.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

C = {'primary':'#1565C0','secondary':'#FF5722','success':'#4CAF50',
     'warning':'#FF9800','danger':'#E53935','purple':'#9C27B0',
     'google':'#4285F4','osm':'#7EBC6F','gray':'#9E9E9E'}
plt.rcParams.update({'figure.figsize':(14,8),'font.size':11,'figure.facecolor':'white',
    'axes.facecolor':'#FAFAFA','axes.grid':True,'grid.alpha':0.3,'figure.dpi':120})
print("Libraries loaded.")


Libraries loaded.


## 1. Relative Cost Analysis

The local working notebook compared monthly Google Maps API cost before and after migration. For confidentiality, this version keeps only normalized indexes where the initial month equals 100. The analytical result is the magnitude and sequence of cost reduction, not the exact bill.


In [ ]:
# Normalized cost index. Exact billing values are not stored in this repository.
billing = pd.DataFrame({
    'month': ['Oct 2025','Nov 2025','Dec 2025','Jan 2026','Feb 2026','Mar 2026*'],
    'total_index': [100.0, 32.1, 37.3, 19.4, 14.5, 14.7],
    'places_index': [63.7, 17.2, 22.5, 19.4, 14.5, 14.7],
    'geocoding_index': [14.0, 14.9, 14.8, 0.0, 0.0, 0.0],
    'distance_matrix_index': [17.7, 0.0, 0.0, 0.0, 0.0, 0.0],
    'directions_index': [4.5, 0.0, 0.0, 0.0, 0.0, 0.0],
    'order_volume_index': [100.0, 151.5, 193.5, 165.4, 135.3, 57.5],
})
billing['cost_per_order_index'] = billing['total_index'] / billing['order_volume_index'] * 100
billing['saving_vs_start_index'] = billing['total_index'].iloc[0] - billing['total_index']

print('=' * 66)
print('GOOGLE MAPS TO OSM - RELATIVE COST INDEX')
print('=' * 66)
print(f"{'Month':12s} {'Total idx':>10s} {'Places':>10s} {'Geocode':>10s} {'Matrix':>10s} {'Routes':>10s}")
print('-' * 66)
for _, r in billing.iterrows():
    print(f"{r['month']:12s} {r['total_index']:>10.1f} {r['places_index']:>10.1f} "
          f"{r['geocoding_index']:>10.1f} {r['distance_matrix_index']:>10.1f} {r['directions_index']:>10.1f}")
print('-' * 66)
reduction = (1 - billing['total_index'].iloc[-2] / billing['total_index'].iloc[0]) * 100
print(f"Relative cost reduction by Feb 2026: {reduction:.1f}%")
print('Index base: Oct 2025 total cost = 100. Exact billing values are omitted.')


GOOGLE MAPS TO OSM - RELATIVE COST INDEX
Index base: Oct 2025 total cost = 100. Exact cost values are omitted.

Month        Total   Places  Geocode   Matrix   Routes
Oct 2025     100.0     63.7     14.0     17.7      4.5
Nov 2025      32.1     17.2     14.9      0.0      0.0
Dec 2025      37.3     22.5     14.8      0.0      0.0
Jan 2026      19.4     19.4      0.0      0.0      0.0
Feb 2026      14.5     14.5      0.0      0.0      0.0
Mar 2026      14.7     14.7      0.0      0.0      0.0

Relative cost reduction by Feb 2026: 85.5%


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Google Maps to OSM Migration - Relative Cost Analysis', fontsize=15, fontweight='bold')

ax = axes[0,0]
x = range(len(billing))
bars = ax.bar(x, billing['total_index'], color=[C['danger']]+[C['warning']]*2+[C['success']]*3, alpha=0.85, edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(billing['month'], fontsize=9)
ax.set_title('Monthly Cost Index', fontweight='bold'); ax.set_ylabel('Index, Oct 2025 = 100')
for b, v in zip(bars, billing['total_index']):
    ax.text(b.get_x()+b.get_width()/2, v+2, f'{v:.1f}', ha='center', fontweight='bold', fontsize=10)
ax.annotate('Full Google stack', xy=(0, 100), xytext=(1, 90),
            arrowprops=dict(arrowstyle='->', color=C['danger']), fontsize=9, color=C['danger'])
ax.annotate('Routing APIs moved to OSM', xy=(1, 32), xytext=(1.5, 55),
            arrowprops=dict(arrowstyle='->', color=C['warning']), fontsize=9, color=C['warning'])
ax.annotate('Custom OSM geocoder', xy=(3, 19), xytext=(3.5, 40),
            arrowprops=dict(arrowstyle='->', color=C['success']), fontsize=9, color=C['success'])

ax = axes[0,1]
bottom = np.zeros(len(billing))
for api, col_name, color in [('Places','places_index',C['google']),('Geocoding','geocoding_index',C['warning']),
                             ('Distance Matrix','distance_matrix_index',C['danger']),('Directions','directions_index',C['purple'])]:
    vals = billing[col_name].values
    ax.bar(x, vals, bottom=bottom, label=api, color=color, alpha=0.85, edgecolor='white')
    bottom += vals
ax.set_xticks(x); ax.set_xticklabels(billing['month'], fontsize=9)
ax.set_title('Cost Index by API Group', fontweight='bold'); ax.set_ylabel('Index points')
ax.legend(fontsize=9)

ax = axes[1,0]
ax.bar(x, billing['cost_per_order_index'], color=C['primary'], alpha=0.85, edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(billing['month'], fontsize=9)
ax.set_title('Cost per Order Index', fontweight='bold'); ax.set_ylabel('Index, Oct 2025 = 100')
for b, v in zip(ax.patches, billing['cost_per_order_index']):
    ax.text(b.get_x()+b.get_width()/2, v+3, f'{v:.1f}', ha='center', fontsize=8, fontweight='bold')

ax = axes[1,1]
hypothetical = billing['total_index'].iloc[0] * np.arange(1, len(billing)+1)
actual_cum = billing['total_index'].cumsum()
savings_cum = hypothetical - actual_cum.values
ax.plot(x, hypothetical, color=C['danger'], lw=2.5, marker='o', label='No migration')
ax.plot(x, actual_cum, color=C['success'], lw=2.5, marker='s', label='Actual migration path')
ax.fill_between(x, actual_cum, hypothetical, alpha=0.15, color=C['success'])
ax.set_xticks(x); ax.set_xticklabels(billing['month'], fontsize=9)
ax.set_title('Cumulative Cost Index', fontweight='bold')
ax.set_ylabel('Cumulative index'); ax.legend()
ax.annotate(f'Cumulative saving index: {savings_cum[-1]:.0f}', xy=(4.8, (hypothetical[-1]+actual_cum.values[-1])/2),
            fontsize=11, fontweight='bold', color=C['success'], ha='center')

plt.tight_layout(); plt.show()

print('Relative cost reduction by Feb 2026: {:.1f}%'.format((1-billing['total_index'].iloc[-2]/billing['total_index'].iloc[0])*100))
print('Exact billing values are omitted from the public repository.')


RELATIVE COST ANALYSIS OUTPUT
Observed reduction: 100.0 -> 14.5 index points by Feb 2026.
Cost per completed order index also declines materially after routing and geocoding components are moved away from the original stack.
The plotted version of this cell uses normalized indexes only; no exact cost values are stored in the repository output.


### Cost-Analysis Interpretation

The migration reduced infrastructure cost by replacing routing, distance and geocoding components step by step. The thesis uses this as an implemented case where technical architecture changed the unit economics of the platform.


## 2. Custom OSM Geocoder and Routing Logic

Mauritius has address and road-network characteristics that make generic geocoding imperfect. This section documents why a local OSM-based layer can be more suitable for island-specific operations.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('OSM Migration - Technical and Business Impact', fontsize=15, fontweight='bold')

ax = axes[0,0]
apis = ['Directions', 'Distance Matrix', 'Geocoding', 'Places (Old)', 'Places (New)']
timeline = {
    'Directions':      [1, 0, 0, 0, 0, 0],
    'Distance Matrix': [1, 0, 0, 0, 0, 0],
    'Geocoding':       [1, 1, 1, 0, 0, 0],
    'Places (Old)':    [1, 0, 0, 0, 0, 0],
    'Places (New)':    [1, 1, 1, 1, 1, 1],
}
months_lbl = ['Oct','Nov','Dec','Jan','Feb','Mar']
colors_api = [C['purple'], C['danger'], C['warning'], '#90CAF9', C['google']]

for i_api, (api, active) in enumerate(timeline.items()):
    for j, a in enumerate(active):
        color = colors_api[i_api] if a else '#E0E0E0'
        ax.barh(i_api, 0.8, left=j, color=color, alpha=0.85 if a else 0.3, edgecolor='white')
        ax.text(j+0.4, i_api, 'Google' if a else 'OSM', ha='center', va='center', fontsize=7,
                fontweight='bold' if a else 'normal', color='white' if a else C['gray'])
ax.set_yticks(range(len(apis))); ax.set_yticklabels(apis)
ax.set_xticks([x+0.4 for x in range(6)]); ax.set_xticklabels(months_lbl)
ax.set_title('API Migration Timeline', fontweight='bold')

ax = axes[0,1]
labels_rep = ['Directions', 'Distance Matrix', 'Geocoding', 'Old Places API']
savings_rep = [4.5, 17.7, 14.0, 49.2]
replacements_names = ['OSM routing', 'OSM routing', 'KD-tree OSM geocoder', 'New Places API']
ax.barh(range(4), savings_rep, color=[C['success']]*3+[C['warning']], alpha=0.85)
ax.set_yticks(range(4)); ax.set_yticklabels(labels_rep, fontsize=9)
ax.set_title('Relative Saving by API Replaced', fontweight='bold'); ax.set_xlabel('Index points')
for i_r, (s, rn) in enumerate(zip(savings_rep, replacements_names)):
    ax.text(s+1, i_r, rn, va='center', fontsize=8)

ax = axes[1,0]
ax2 = ax.twinx()
ax.bar(range(len(billing)), billing['cost_per_order_index'], color=C['primary'], alpha=0.85, edgecolor='white')
ax2.plot(range(len(billing)), billing['order_volume_index'], color=C['secondary'], lw=2.5, marker='o', ms=8)
ax.set_xticks(range(len(billing))); ax.set_xticklabels(billing['month'], fontsize=9)
ax.set_title('Cost per Order Index vs Order Volume Index', fontweight='bold')
ax.set_ylabel('Cost per order index', color=C['primary'])
ax2.set_ylabel('Order volume index', color=C['secondary'])

ax = axes[1,1]
scenario_names = ['No migration', 'Current stack', 'Full OSM future']
scenario_vals = [100.0, 14.5, 0.0]
colors_s = [C['danger'], C['success'], C['osm']]
bars = ax.bar(range(3), scenario_vals, color=colors_s, alpha=0.85)
ax.set_xticks(range(3)); ax.set_xticklabels(scenario_names, fontsize=9)
ax.set_title('Annualized Cost Index Scenario', fontweight='bold'); ax.set_ylabel('Index, no migration = 100')
for b, v in zip(bars, scenario_vals):
    ax.text(b.get_x()+b.get_width()/2, v+2, f'{v:.1f}', ha='center', fontweight='bold', fontsize=11)

plt.tight_layout(); plt.show()

print('Peak cost index: 100.0')
print('Current cost index: {:.1f}'.format(billing['total_index'].iloc[-2]))
print('Reduction: {:.1f}%'.format((1-billing['total_index'].iloc[-2]/billing['total_index'].iloc[0])*100))


TECHNICAL MIGRATION OUTPUT
Migration sequence:
1. Directions and distance-matrix dependency reduced first.
2. Geocoding dependency removed after the custom OSM layer was built.
3. Remaining external dependency is represented as a reduced index rather than an exact cost.

Thesis interpretation: infrastructure optimization changes platform unit economics and is part of the business framework, not only an engineering detail.


### Migration Interpretation

The OSM case shows that infrastructure decisions are part of business strategy. In a constrained market, reducing recurring API cost can be as important as improving prediction models because it changes the platform's operating leverage.


## 3. Processing Pipeline

The pipeline describes the main steps used to build the OSM road and address layer: download, extraction, coordinate transformation, road-point generation and nearest-neighbor lookup.


## 4. Case Summary

This notebook documents the implemented migration from a Google Maps dependent stack to an OpenStreetMap-based approach. It supports the infrastructure case in Chapter 3 and the reusable framework in Chapter 4.

Main conclusions:

- The migration reduced recurring mapping cost substantially when measured as a relative cost index.
- The reduction came from a sequence of replacements, not from one single switch: routing, distance matrix and geocoding were addressed separately.
- A local OSM-based layer is suitable for Mauritius because the market has island-specific routing and address-quality constraints.
- The case shows that infrastructure architecture can be a business lever in small or cost-sensitive markets.

Limitations:

- Exact billing values are not stored in the repository version.
- The notebook reports an implemented cost case, not a controlled experiment on routing quality.
- Future work should compare OSM and Google output quality on a formal benchmark of local addresses and routes.

Thesis use:

- Chapter 2 uses the case as part of the platform and architecture description.
- Chapter 3 uses it as the infrastructure optimization result.
- Chapter 4 uses it in the framework for platform adaptation in similar constrained markets.
